# Análise Comparativa — Todos os PCAPs

Comparação direta entre tráfego benigno e 6 ataques SOME/IP.  
Estatísticas pré-extraídas via `tshark` e salvas em `pcap_stats.json`.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

import json, os
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings; warnings.filterwarnings('ignore')

STATS_FILE = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                           'pcap_stats.json')
with open(STATS_FILE, encoding='utf-8') as f:
    results = json.load(f)
LABELS = list(results.keys())

BG='#0f1117'; PANEL='#161b2e'; GRID='#2a3550'
TEXT='#d0d8f0'; A='#4e7fff'; G='#7bff9c'
T2='#7bffd9'; Y='#ffd97b'; P='#c07bff'
O='#ff9d7b'; R='#ff7b7b'
COLORS=[G,R,Y,A,T2,P,O]

LB=dict(paper_bgcolor=BG,plot_bgcolor=PANEL,
        font=dict(color=TEXT,family='monospace'),
        title_font=dict(color=TEXT,size=15))

_first=True
def show(fig):
    global _first
    display(HTML(fig.to_html(full_html=False,
                             include_plotlyjs='cdn' if _first else False)))
    _first=False

print(f'Carregados: {LABELS}')

Carregados: ['benign', 'dos_noti', 'fuzzy(1)', 'fuzzy(2)', 'fuzzy(3)', 'mitm_multi', 'mitm_single']


## 1. Visão Geral — Métricas por PCAP

In [2]:
rows=[(l,
       f"{r['n_frame']:,}",
       f"{r['dur']:.0f}s",
       f"{r['avg_pps']:,.0f}",
       f"{r['peak_pps']:,}",
       f"{r['n_si']:,}  ({r['n_si']/r['n_frame']*100:.1f}%)",
       f"{r['n_sd']:,}",
       str(r['n_ips']),
       str(r['ttl']),
      ) for l,r in results.items()]

headers=['PCAP','Frames','Duracao','Avg pkt/s','Peak pkt/s',
         'SOME/IP','SOME/IP-SD','IPs unicos','TTLs']
cols=list(zip(*rows))

fig=go.Figure(go.Table(
    header=dict(values=headers,fill_color=PANEL,
                font=dict(color=A,size=12),align='left'),
    cells=dict(values=[list(c) for c in cols],
               fill_color=[[BG if i%2==0 else PANEL for i in range(len(rows))]
                           for _ in headers],
               font=dict(color=TEXT,size=11),align='left')))
fig.update_layout(**LB,title='Visão geral por PCAP',height=320)
show(fig)

## 2. Composição do Tráfego por Protocolo

In [3]:
protos  =['SOME/IP (TCP)','TCP ctrl/ACK','SOME/IP-SD','SOME/IP (UDP)','IGMPv3','ARP']
keys    =['n_si_tcp',    'n_tcp_c',     'n_sd',      'n_si_udp',    'n_igmp','n_arp']
pcolors =[A,             R,             Y,            T2,            G,       P]

fig=go.Figure()
for proto,key,col in zip(protos,keys,pcolors):
    vals=[results[l][key] for l in LABELS]
    pcts=[v/results[l]['n_frame']*100 for l,v in zip(LABELS,vals)]
    fig.add_trace(go.Bar(
        name=proto, x=LABELS, y=pcts,
        marker_color=col,
        hovertemplate=f'{proto}: %{{y:.1f}}%<extra></extra>'))

fig.update_layout(**LB,
    title='Composicao por protocolo (% frames)',
    barmode='stack', height=460,
    xaxis=dict(title='PCAP',gridcolor=GRID),
    yaxis=dict(title='% frames',gridcolor=GRID),
    legend=dict(bgcolor=PANEL,bordercolor=GRID))
show(fig)

## 3. Distribuição de Message Type (SOME/IP)

In [4]:
mt_order=['NOTIFICATION','REQUEST','RESPONSE','REQUEST_NO_RETURN',
          'ERROR','REQUEST_ACK','RQST_NRET_ACK','NOTIF_ACK']
mt_col  =[R,A,G,Y,O,T2,P,A]

fig=go.Figure()
for mt,col in zip(mt_order,mt_col):
    vals=[]
    for l in LABELS:
        n_si=results[l]['n_si']
        cnt=results[l]['mt_counts'].get(mt,0)
        vals.append(cnt/n_si*100 if n_si>0 else 0)
    if any(v>0 for v in vals):
        fig.add_trace(go.Bar(
            name=mt, x=LABELS, y=vals,
            marker_color=col,
            hovertemplate=f'{mt}: %{{y:.1f}}%<extra></extra>'))

fig.update_layout(**LB,
    title='Message Type — % dos frames SOME/IP por PCAP',
    barmode='stack', height=460,
    xaxis=dict(title='PCAP',gridcolor=GRID),
    yaxis=dict(title='% SOME/IP frames',gridcolor=GRID),
    legend=dict(bgcolor=PANEL,bordercolor=GRID))
show(fig)

print(f'{"PCAP":<14}',end='')
for mt in mt_order: print(f'{mt[:10]:>11}',end='')
print()
print('-'*104)
for l in LABELS:
    n_si=results[l]['n_si']
    print(f'{l:<14}',end='')
    for mt in mt_order:
        cnt=results[l]['mt_counts'].get(mt,0)
        pct=cnt/n_si*100 if n_si>0 else 0
        print(f'{pct:>10.1f}%',end='')
    print()

PCAP           NOTIFICATI    REQUEST   RESPONSE REQUEST_NO      ERROR REQUEST_AC RQST_NRET_  NOTIF_ACK
--------------------------------------------------------------------------------------------------------
benign              99.7%       0.1%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%
dos_noti            99.7%       0.2%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%
fuzzy(1)            99.7%       0.1%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%
fuzzy(2)            99.5%       0.2%       0.2%       0.0%       0.0%       0.0%       0.0%       0.0%
fuzzy(3)            99.7%       0.1%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%
mitm_multi          99.8%       0.1%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%
mitm_single         99.8%       0.1%       0.1%       0.0%       0.0%       0.0%       0.0%       0.0%


## 4. Service IDs — Heatmap de Alvos por Ataque

In [5]:
all_svcs=sorted(set(s for l in LABELS for s in results[l]['svc_counts']))

z=[]
for l in LABELS:
    n_si=results[l]['n_si']
    row=[results[l]['svc_counts'].get(s,0)/n_si*100 if n_si>0 else 0
         for s in all_svcs]
    z.append(row)

text_z=[[f'{v:.0f}%' if v>0.5 else '' for v in row] for row in z]

fig=go.Figure(go.Heatmap(
    z=z, x=all_svcs, y=LABELS,
    colorscale=[[0,'#0f1117'],[0.01,A],[0.3,Y],[1.0,R]],
    hovertemplate='PCAP: %{y}<br>Service: %{x}<br>%{z:.1f}%<extra></extra>',
    text=text_z, texttemplate='%{text}',
    showscale=True))
fig.update_layout(**LB,
    title='Service ID — % dos frames SOME/IP (top 10 por PCAP)',
    height=380,
    xaxis=dict(title='Service ID',gridcolor=GRID,tickangle=30),
    yaxis=dict(title='PCAP',gridcolor=GRID))
show(fig)

## 5. Volume e Taxa de Tráfego

In [6]:
fig=make_subplots(rows=1,cols=2,
    subplot_titles=['Total de Frames','Taxa Média (pkt/s)'])

frames_v=[results[l]['n_frame'] for l in LABELS]
pps_v   =[results[l]['avg_pps']  for l in LABELS]

fig.add_trace(go.Bar(
    x=LABELS, y=frames_v, marker_color=COLORS,
    text=[f'{v:,.0f}' for v in frames_v], textposition='outside',
    hovertemplate='%{x}: %{y:,}<extra></extra>'),row=1,col=1)

fig.add_trace(go.Bar(
    x=LABELS, y=pps_v, marker_color=COLORS,
    text=[f'{v:,.0f}' for v in pps_v], textposition='outside',
    hovertemplate='%{x}: %{y:,.0f} pkt/s<extra></extra>'),row=1,col=2)

fig.update_layout(**LB,title='Volume e taxa de tráfego',height=440,showlegend=False,
    xaxis =dict(gridcolor=GRID,tickangle=30),
    xaxis2=dict(gridcolor=GRID,tickangle=30),
    yaxis =dict(title='Frames',gridcolor=GRID),
    yaxis2=dict(title='pkt/s',gridcolor=GRID))
show(fig)

## 6. Série Temporal — Intensidade SOME/IP (pkt/s)

In [7]:
attacks=[l for l in LABELS if l!='benign']

fig=make_subplots(rows=2,cols=1,
    subplot_titles=['Ataques — SOME/IP pkt/s','Benigno — SOME/IP pkt/s'],
    vertical_spacing=0.12)

for label,col in zip(attacks,COLORS[1:]):
    ts=results[label]['ts']
    t_vals=[x[0] for x in ts]
    c_vals=[x[1] for x in ts]
    fig.add_trace(go.Scatter(
        x=t_vals, y=c_vals, name=label, mode='lines',
        line=dict(color=col,width=1),
        hovertemplate=f'{label}: %{{y}} pkt/s @ t=%{{x}}s<extra></extra>'),
    row=1,col=1)

if 'benign' in results:
    ts_b=results['benign']['ts']
    fig.add_trace(go.Scatter(
        x=[x[0] for x in ts_b], y=[x[1] for x in ts_b],
        name='benign', mode='lines',
        line=dict(color=G,width=1),
        hovertemplate='benign: %{y} pkt/s @ t=%{x}s<extra></extra>'),
    row=2,col=1)

fig.update_layout(**LB,title='Intensidade SOME/IP pkt/s',height=640,
    xaxis =dict(title='t (s)',gridcolor=GRID),
    xaxis2=dict(title='t (s)',gridcolor=GRID),
    yaxis =dict(title='pkt/s',gridcolor=GRID),
    yaxis2=dict(title='pkt/s',gridcolor=GRID),
    legend=dict(bgcolor=PANEL,bordercolor=GRID))
show(fig)

print('Pico por PCAP:')
for l in LABELS:
    print(f'  {l:<14} {results[l]["peak_pps"]:>8,} pkt/s')

Pico por PCAP:
  benign              855 pkt/s
  dos_noti            693 pkt/s
  fuzzy(1)            861 pkt/s
  fuzzy(2)            502 pkt/s
  fuzzy(3)            876 pkt/s
  mitm_multi          999 pkt/s
  mitm_single         897 pkt/s


## 7. TTL — Anomalias por PCAP

In [8]:
print(f'{"PCAP":<16} {"TTLs observados"}')
print('-'*55)
for l in LABELS:
    ttls=results[l]['ttl']
    flag='  << TTL=1 (possivel spoof)' if 1 in ttls else ''
    print(f'{l:<16} {ttls}{flag}')

PCAP             TTLs observados
-------------------------------------------------------
benign           [1, 64]  << TTL=1 (possivel spoof)
dos_noti         [1, 64]  << TTL=1 (possivel spoof)
fuzzy(1)         [1, 64]  << TTL=1 (possivel spoof)
fuzzy(2)         [1, 64]  << TTL=1 (possivel spoof)
fuzzy(3)         [1, 64]  << TTL=1 (possivel spoof)
mitm_multi       [1, 64]  << TTL=1 (possivel spoof)
mitm_single      [1, 64]  << TTL=1 (possivel spoof)


## 8. Características de Ataque Detectáveis

Indicadores derivados diretamente dos dados — comparação com baseline benigno.

In [9]:
# ── Baseline benigno ────────────────────────────────────────────────────
B = results['benign']
b_sd_ratio   = B['n_sd'] / B['n_si'] * 100        # SD% normal
b_0x1001     = B['svc_counts'].get('0x1001', 0)   # vol normal do svc principal
b_n_ips      = B['n_ips']
b_svcs       = set(B['svc_counts'].keys())
attacks      = [l for l in LABELS if l != 'benign']

# ── Calcular indicadores por ataque ──────────────────────────────────────
rows = []
for l in attacks:
    r = results[l]
    sd_ratio   = r['n_sd'] / r['n_si'] * 100 if r['n_si'] > 0 else 0
    v_0x1001   = r['svc_counts'].get('0x1001', 0)
    drop_0x1001= (1 - v_0x1001 / b_0x1001) * 100 if b_0x1001 > 0 else 0
    new_svcs   = set(r['svc_counts'].keys()) - b_svcs
    extra_ips  = r['n_ips'] - b_n_ips
    rows.append(dict(label=l, sd_ratio=sd_ratio, drop_0x1001=drop_0x1001,
                     new_svcs=new_svcs, extra_ips=extra_ips))

# ── Heatmap de severidade dos indicadores ────────────────────────────────
# Cada indicador normalizado 0-1 (0=normal, 1=máximo desvio observado)
ind_labels = ['SD flood\n(ratio SD%)', 'Disrupção\n0x1001', 'Serviços\ndesconhecidos', 'IPs extras']

max_sd  = max(r['sd_ratio']  for r in rows)
max_drop= max(r['drop_0x1001'] for r in rows)
max_svcs= max(len(r['new_svcs']) for r in rows) or 1
max_ips = max(r['extra_ips']  for r in rows) or 1

z, text_z = [], []
for r in rows:
    row_z = [
        r['sd_ratio']    / max_sd   if max_sd   > 0 else 0,
        r['drop_0x1001'] / max_drop if max_drop > 0 else 0,
        len(r['new_svcs']) / max_svcs,
        max(r['extra_ips'], 0) / max_ips,
    ]
    row_t = [
        f"{r['sd_ratio']:.1f}%",
        f"-{r['drop_0x1001']:.0f}%",
        ', '.join(sorted(r['new_svcs'])) if r['new_svcs'] else '—',
        f"+{r['extra_ips']}" if r['extra_ips'] > 0 else '0',
    ]
    z.append(row_z)
    text_z.append(row_t)

fig = go.Figure(go.Heatmap(
    z=z,
    x=ind_labels,
    y=[r['label'] for r in rows],
    colorscale=[[0,'#0f1117'],[0.2,A],[0.6,Y],[1.0,R]],
    zmin=0, zmax=1,
    text=text_z, texttemplate='%{text}',
    hovertemplate='%{y} — %{x}<br>valor: %{text}<extra></extra>',
    showscale=False))
fig.update_layout(**LB,
    title='Indicadores de ataque — severidade relativa ao baseline benigno',
    height=340,
    xaxis=dict(side='top', gridcolor=GRID),
    yaxis=dict(gridcolor=GRID))
show(fig)

# ── Tabela textual detalhada ──────────────────────────────────────────────
print(f'Baseline benigno:')
print(f'  SD ratio     : {b_sd_ratio:.1f}%')
print(f'  Vol 0x1001   : {b_0x1001:,} frames')
print(f'  IPs          : {b_n_ips}')
print(f'  Services     : {sorted(b_svcs)}')
print()
print(f'{"PCAP":<14} {"SD%":>8} {"SD anomalia":>12} {"Drop 0x1001":>12} {"Svcs novos":>14} {"IPs extras":>10}')
print('-'*74)
for r in rows:
    sd_flag = '<<ALTO' if r['sd_ratio'] > b_sd_ratio * 3 else '     '
    dp_flag = f'-{r["drop_0x1001"]:.0f}%' if r['drop_0x1001'] > 50 else f'-{r["drop_0x1001"]:.0f}%'
    dp_mark = '<<DISRUP' if r['drop_0x1001'] > 50 else ''
    print(f'{r["label"]:<14} {r["sd_ratio"]:>7.1f}% {sd_flag:>12} {dp_flag:>10} {dp_mark:>14} {r["extra_ips"]:>10}')
    if r['new_svcs']:
        print(f'  Serv. desconhecidos: {sorted(r["new_svcs"])}')

# ── Resumo das características observáveis ────────────────────────────────
print()
print('=' * 68)
print('RESUMO — Características de Ataque Detectáveis')
print('=' * 68)
print()
print('1. SD FLOOD (SOME/IP-SD ratio > 3x baseline)')
print(f'   Baseline: {b_sd_ratio:.1f}%  |  Limiar: {b_sd_ratio*3:.1f}%')
for r in rows:
    mark = '  *** DETECTADO' if r['sd_ratio'] > b_sd_ratio * 3 else ''
    print(f'   {r["label"]:<14} {r["sd_ratio"]:>6.1f}%{mark}')
print()
print('2. DISRUPÇÃO DE SERVIÇO (queda > 50% no volume do 0x1001)')
print(f'   Baseline 0x1001: {b_0x1001:,} frames')
for r in rows:
    mark = '  *** DETECTADO' if r['drop_0x1001'] > 50 else ''
    print(f'   {r["label"]:<14} -{r["drop_0x1001"]:>5.1f}% de queda{mark}')
print()
print('3. SERVIÇO DESCONHECIDO (service ID ausente no benigno)')
for r in rows:
    if r['new_svcs']:
        print(f'   {r["label"]:<14} {sorted(r["new_svcs"])}  *** DETECTADO')
    else:
        print(f'   {r["label"]:<14} nenhum')
print()
print('4. IP EXTRA (novos participantes vs baseline)')
for r in rows:
    mark = '  *** DETECTADO' if r['extra_ips'] > 0 else ''
    print(f'   {r["label"]:<14} +{r["extra_ips"]} IP(s){mark}')

Baseline benigno:
  SD ratio     : 1.7%
  Vol 0x1001   : 492,058 frames
  IPs          : 9
  Services     : ['0x1001', '0x1002', '0x1003', '0xffff']

PCAP                SD%  SD anomalia  Drop 0x1001     Svcs novos IPs extras
--------------------------------------------------------------------------
dos_noti          31.4%       <<ALTO       -86%       <<DISRUP          1
fuzzy(1)           2.2%                     -0%                         1
fuzzy(2)           3.9%                    -93%       <<DISRUP          1
fuzzy(3)           1.7%                    --3%                         0
mitm_multi        15.0%       <<ALTO       -31%                         2
  Serv. desconhecidos: ['0x100b']
mitm_single       42.7%       <<ALTO       -89%       <<DISRUP          1

RESUMO — Características de Ataque Detectáveis

1. SD FLOOD (SOME/IP-SD ratio > 3x baseline)
   Baseline: 1.7%  |  Limiar: 5.1%
   dos_noti         31.4%  *** DETECTADO
   fuzzy(1)          2.2%
   fuzzy(2)          3.9%

## 9. O que foi possível caracterizar

In [10]:
# ── Matriz de detecção ──────────────────────────────────────────────────
# Status por ataque e por indicador
# SIM=2 (verde), PARCIAL=1 (amarelo), NAO=0 (vermelho)
B = results['benign']
b_sd  = B['n_sd'] / B['n_si'] * 100
b_001 = B['svc_counts'].get('0x1001', 0)
b_ips = B['n_ips']
b_svcs= set(B['svc_counts'].keys())
attacks=[l for l in LABELS if l!='benign']

SIM='SIM'; PARC='PARCIAL'; NAO='NÃO'
GREEN='#1a3d1a'; YELLOW='#3d3300'; RED='#3d0f0f'
TGREEN='#7bff9c'; TYELLOW='#ffd97b'; TRED='#ff7b7b'

matrix = []
for l in attacks:
    r = results[l]
    sd_ratio  = r['n_sd']/r['n_si']*100 if r['n_si']>0 else 0
    drop_001  = (1 - r['svc_counts'].get('0x1001',0)/b_001)*100 if b_001>0 else 0
    new_svcs  = set(r['svc_counts'].keys()) - b_svcs
    extra_ips = r['n_ips'] - b_ips

    sd_det  = SIM  if sd_ratio > b_sd*3   else (PARC if sd_ratio > b_sd*1.5 else NAO)
    dp_det  = SIM  if drop_001 > 50        else (PARC if drop_001 > 20        else NAO)
    sv_det  = SIM  if new_svcs             else NAO
    ip_det  = SIM  if extra_ips > 0        else NAO
    # detectavel por metadados = pelo menos 1 SIM
    overall = SIM  if SIM in [sd_det,dp_det,sv_det,ip_det] \
              else (PARC if PARC in [sd_det,dp_det,sv_det,ip_det] else NAO)
    matrix.append([l, sd_det, dp_det, sv_det, ip_det, overall])

cols_h = ['Ataque','SD Flood','Disrupção\nde Serviço',
          'Serviço\nInjetado','IP Extra','Detectável por\nMetadados']

def cell_color(v):
    return GREEN if v==SIM else (YELLOW if v==PARC else RED)
def text_color(v):
    return TGREEN if v==SIM else (TYELLOW if v==PARC else TRED)

fill_cols = [[BG]*len(matrix)]  # coluna label sempre escura
font_cols = [[TEXT]*len(matrix)]
for ci in range(1, len(cols_h)):
    fill_cols.append([cell_color(row[ci]) for row in matrix])
    font_cols.append([text_color(row[ci]) for row in matrix])

fig = go.Figure(go.Table(
    header=dict(values=cols_h,
                fill_color=PANEL,
                font=dict(color=A, size=12),
                align='center', height=40),
    cells=dict(
        values=[[row[i] for row in matrix] for i in range(len(cols_h))],
        fill_color=fill_cols,
        font=dict(color=font_cols, size=12, family='monospace'),
        align='center', height=36)))
fig.update_layout(**LB,
    title='Matriz de detecção — o que é visível nos metadados de rede',
    height=320)
show(fig)

# ── Modelo de ameaça ─────────────────────────────────────────────────────
threat_html = '''
<div style="background:#161b2e;border-left:4px solid #4e7fff;
            padding:18px 24px;margin:18px 0;font-family:monospace;
            color:#d0d8f0;border-radius:4px;">
<h3 style="color:#4e7fff;margin-top:0">Modelo de Ameaça — o que os dados implicam</h3>
<p><b style="color:#7bff9c">Vetor de acesso:</b> atacante já está <i>dentro</i> da rede
   (IPs no mesmo segmento 172.18.0.x). Não é intrusão externa.</p>
<p><b style="color:#7bff9c">Ausência de autenticação:</b> qualquer nó pode anunciar
   qualquer Service ID. O 0x100b do mitm_multi aparece sem resistência na rede.</p>
<p><b style="color:#7bff9c">SD como ponto fraco:</b> SOME/IP-SD não valida quem pode
   fazer Offer de um serviço. Três ataques (dos_noti, mitm_single, mitm_multi)
   exploram isso — SD ratio dispara até 42.7% vs 1.7% no benigno.</p>
<p><b style="color:#7bff9c">DoS por substituição, não por volume:</b> dos_noti mantém
   a mesma taxa de pacotes (~1300 pkt/s). O ataque desloca tráfego legítimo,
   não satura a banda.</p>
<p><b style="color:#ffd97b">Fuzzy(1) e fuzzy(3) são invisíveis por metadados:</b>
   estatísticas globais idênticas ao benigno. O ataque está no conteúdo dos
   campos SOME/IP — requer inspeção de payload para ser detectado.</p>
</div>
'''
display(HTML(threat_html))

# ── O que não foi possível determinar ────────────────────────────────────
limits_html = '''
<div style="background:#1a1200;border-left:4px solid #ffd97b;
            padding:18px 24px;margin:18px 0;font-family:monospace;
            color:#d0d8f0;border-radius:4px;">
<h3 style="color:#ffd97b;margin-top:0">Limitações — o que requer inspeção de payload</h3>
<ul style="margin:0;padding-left:20px">
<li><b>Fuzzy(1) e fuzzy(3):</b> indistinguíveis do benigno por distribuição de pacotes.
    O ataque está nos valores dos campos SOME/IP (Method ID, Return Code, payload bytes).</li>
<li><b>Timestamp de início do ataque:</b> não há ground truth de quando cada ataque começa
    dentro do PCAP — a janela de ataque é desconhecida.</li>
<li><b>Intenção vs ruído:</b> a queda de 0x1001 pode ser consequência do ataque ou
    comportamento normal do sistema naquele momento.</li>
</ul>
</div>
'''
display(HTML(limits_html))